# LFM3 text inference with Transformers

Use a Python 3.12 GPU runtime. This example uses Transformers 5.23.0 and the checkpoint-provided LFM3 code. Review that code before enabling remote code.


In [ ]:
%pip install "transformers==5.23.0" torch accelerate


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Change the ID and preset together for an older text model.
model_id = "LiquidAI/LFM3-1.7B"
remote_code = model_id.startswith("LiquidAI/LFM3")
model = AutoModelForCausalLM.from_pretrained(
    model_id, trust_remote_code=remote_code, device_map="auto", dtype="bfloat16",
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=remote_code)
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": "What is the capital of France?"}],
    add_generation_prompt=True, tokenize=True,
    return_tensors="pt", return_dict=True,
).to(model.device)
output = model.generate(
    **inputs, do_sample=True, temperature=0.1, top_k=50,
    repetition_penalty=1.1, max_new_tokens=2048,
)
response = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)


## Older models

The same loading/template/generation flow was checked with LFM2.5-1.2B-Instruct and LFM2-1.2B. For LFM2.5, set `repetition_penalty=1.05`. For original LFM2, use `temperature=0.3`, `min_p=0.15`, and `repetition_penalty=1.05` instead of the LFM3 preset. Check deprecation notices before starting a new deployment.

The raw LFM3 decode includes reasoning. A token cap can stop before the final answer. Follow [Thinking](https://docs.liquid.ai/lfm/key-concepts/thinking) for controls and parsing, and use the [model page](https://docs.liquid.ai/lfm/models/lfm3-1.7b) for current settings.
